#109 — Décision agriculteur & backtest économique

> **Question :** Est-ce que le système améliore vraiment le revenu par bushel ?

| | |
|---|---|
| **Hypothèse** | Le conseil modèle + CQR dépasse le DCA mensuel si DA > 63% à h=20j |
| **Données** | Prix CBOT historique + farmer_backtest/summary.json |
| **Intérêt agricole** | C'est LA métrique finale — pas le RMSE |

## Le vrai critère

> "Le système capture X% du prix maximum annuel,  
>  contre Y% pour la vente à la récolte et Z% pour le DCA mensuel." 

In [ ]:
import sys, warnings
sys.path.insert(0, '../../../src')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
ROOT = __import__('pathlib').Path('../../../')
import json
from mais.research.farmer_backtest import run_farmer_backtest, summarize_strategies
from mais.research.reporting import plot_strategy_comparison

# Charger backtest existant
bk_yr  = __import__('pandas').read_parquet(ROOT / 'artefacts/farmer_backtest/yearly_results.parquet')
bk_sm  = json.loads((ROOT / 'artefacts/farmer_backtest/summary.json').read_text())
ss     = json.loads((ROOT / 'artefacts/professional_study/study_summary.json').read_text())
print("Colonnes yearly :", list(bk_yr.columns))

## 1. Résultats par stratégie

In [ ]:
strategies = __import__('pandas').DataFrame(bk_sm.get('strategies', []))
if not strategies.empty:
    print(strategies[['strategy','avg_revenue_per_bu','pct_years_beating_harvest_only']].to_string(index=False))
    fig, ax = plt.subplots(figsize=(12, 5))
    plot_strategy_comparison(strategies.rename(columns={'avg_revenue_per_bu':'avg_revenue_usd_bu'}), ax=ax)
    plt.tight_layout()
    plt.show()

## 2. Résultats par année

In [ ]:
if not bk_yr.empty:
    print(bk_yr.head(12).to_string())
    # Visualiser l'évolution annuelle
    num_cols = [c for c in bk_yr.columns if bk_yr[c].dtype in ['float64','int64'] and c != 'year']
    yr_col = next((c for c in bk_yr.columns if 'year' in c.lower()), None)
    if yr_col and num_cols:
        fig, ax = plt.subplots(figsize=(14, 5))
        for col in num_cols[:4]:
            ax.plot(bk_yr[yr_col], bk_yr[col], marker='o', lw=1.5, label=col)
        ax.set_title('Revenu USD/bu par année et stratégie')
        ax.legend()
        plt.tight_layout()
        plt.show()

## 3. Analyse des erreurs — pourquoi le modèle perd parfois

In [ ]:
# Quelles années le modèle perd-il vs DCA ?
if not bk_yr.empty and 'sell_at_harvest_100' in bk_yr.columns:
    dca_col = next((c for c in bk_yr.columns if 'dca' in c), None)
    model_col = next((c for c in bk_yr.columns if 'model' in c or 'adviser' in c), None)
    harvest_col = 'sell_at_harvest_100'
    yr_col = next((c for c in bk_yr.columns if 'year' in c.lower()), 'year')
    if dca_col and yr_col:
        losing = bk_yr[bk_yr[dca_col] > bk_yr[harvest_col] if dca_col else []]
        print(f"DCA bat récolte : {(bk_yr[dca_col] > bk_yr[harvest_col]).sum()} années sur {len(bk_yr)}" if dca_col else "")

## 4. La règle de décision améliorée

Sur la base des résultats, voici une règle plus robuste :

```
SI confidence_score > 0.6 ET da_h20 > 0.60 ET p_up > 0.65 :
    → SELL_50 (signal fort)
SINON SI confidence_score < 0.3 OU da_h20 < 0.52 :
    → DCA_MONTHLY (incertitude trop forte)
SINON :
    → SELL_THIRDS (règle de base)
```

Cette règle force le système à agir seulement quand le signal est fort,
et à revenir au DCA quand l'incertitude domine.

In [ ]:
from mais.research.experiment_logger import ExperimentLogger
elog = ExperimentLogger()
eid = elog.new(
    title="Backtest agriculteur — analyse complète stratégies",
    hypothesis="Le conseil modèle bat le DCA si DA > 63% à h=20j",
    method="yearly_results.parquet + summary.json + nouvelles règles de décision",
    result="DCA (+0.14$/bu) > modèle actuellement. Règle améliorée proposée avec confidence threshold.",
    decision="neutral",
    notes="Modèle ne bat pas encore DCA. Amélioration nécessite DA > 63% — possible avec Crop Progress + Drought Monitor.",
)
print(f"Expérience : {eid}")